In [4]:
import seaborn as sns
import pandas as pd
import numpy as np

In [32]:
#orginele dataset inlezen
df = pd.read_csv('Leesmateriaal.csv')
df.head()

,Titel,Auteur,Korte omschrijving,Unnamed: 3,Type materiaal,Niveau,Themas_boek,materiaal_id,genre,onderwerp,...,interessecluster,uitgever,publicatiejaar,beschikbaar_formaat,cataloguscode,interne_batchcode,administratieve_prioriteit,doelgroep_extra,genre_kopie,Gemiddelde beoordeling
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,NaN,boek,3F,WOII;onderduik,LM-0001,fictie,WOII,...,VERHALEN,Educatief Media,2025,PDF,CAT-00001,B1,Hoog,NaN,fictie,3.4
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,NaN,Boek,3F,spanning;stalking,LM-0002,fictie,spanning,...,verhalen,Leeswereld,2005,Digitaal,CAT-00002,B2,hoog,NT2,fictie,2.5
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",NaN,Boek,2F,liefde;relaties,LM-0003,fictie,liefde,...,verhalen,NaN,2025,Digitaal,CAT-00003,B2,Midden,NT2,fictie,3.8
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,NaN,Boek,3F,ontmoeting;verbondenheid,LM-0004,fictie,ontmoeting,...,verhalen,Uitgeverij Noord,2018,Digitaal,CAT-00004,batch-2,Laag,NT2,fictie,3.9
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,NaN,Boek,2F,geheimen;schuld,LM-0005,fictie,geheimen,...,verhalen,Uitgeverij Noord,NaN,online,CAT-00005,B3,Laag,MBO/NT2,fictie,3.1


In [33]:
#Alle overige kolommen verwijderen die niet relevant zijn voor de analyse
df = df.drop(columns=['Unnamed: 3', 'materiaal_id', 'onderwerp', 'trefwoorden', 'bron_url', 
                      'gegevensstatus', 'woordenaantal_omschrijving', 'gem_woordlengte', 
                      'gem_zinslengte_woorden', 'aandeel_lange_woorden_pct', 'aantal_zinnen', 'leescomplexiteit_score', 
                      'interessecluster', 'uitgever', 'publicatiejaar', 'beschikbaar_formaat', 'interne_batchcode', 
                      'administratieve_prioriteit', 'genre_kopie', 'Gemiddelde beoordeling'])

df.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,Themas_boek,genre,taal,aantal_paginas,geschatte_leestijd_min,doelgroep,cataloguscode,doelgroep_extra
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,boek,3F,WOII;onderduik,fictie,Nederlands,265.0,556.0,MBO,CAT-00001,NaN
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,spanning;stalking,fictie,Nederlands,171.0,184.0,MBO,CAT-00002,NT2
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,liefde;relaties,fictie,Nederlands,219.0,662.0,MBO/NT2,CAT-00003,NT2
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,Boek,3F,ontmoeting;verbondenheid,fictie,Nederlands,288.0,748.0,MBO,CAT-00004,NT2
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,Boek,2F,geheimen;schuld,fictie,Nederlands,358.0,196.0,MBO/NT2,CAT-00005,MBO/NT2


In [13]:
#Merge de kolommen 'doelgroep' en 'doelgroep_extra' tot één kolom 'doelgroep', waarbij dubbele waarden worden verwijderd
def merge_doelgroep(row):
    unieke_doelgroepen = []

    for kolom in ["doelgroep", "doelgroep_extra"]:
        waarde = row[kolom]

        if pd.notna(waarde):
            for doelgroep in str(waarde).split("/"):
                doelgroep = doelgroep.strip()

                if doelgroep and doelgroep not in unieke_doelgroepen:
                    unieke_doelgroepen.append(doelgroep)

    return "/".join(unieke_doelgroepen)


df["doelgroep"] = df.apply(merge_doelgroep, axis=1)
df = df.drop(columns=["doelgroep_extra"])
df.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,Themas_boek,genre,taal,aantal_paginas,geschatte_leestijd_min,doelgroep,cataloguscode
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,boek,3F,WOII;onderduik,fictie,Nederlands,265.0,556.0,MBO,CAT-00001
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,spanning;stalking,fictie,Nederlands,171.0,184.0,MBO/NT2,CAT-00002
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,liefde;relaties,fictie,Nederlands,219.0,662.0,MBO/NT2,CAT-00003
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,Boek,3F,ontmoeting;verbondenheid,fictie,Nederlands,288.0,748.0,MBO/NT2,CAT-00004
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,Boek,2F,geheimen;schuld,fictie,Nederlands,358.0,196.0,MBO/NT2,CAT-00005


In [ ]:
#Missing values in de "Auteur" kolom vullen met "Onbekend"
df["Auteur"] = df["Auteur"].fillna("Onbekend")
df.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,Themas_boek,genre,taal,aantal_paginas,geschatte_leestijd_min,doelgroep,cataloguscode
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,boek,3F,WOII;onderduik,fictie,Nederlands,265.0,556.0,MBO,CAT-00001
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,spanning;stalking,fictie,Nederlands,171.0,184.0,MBO/NT2,CAT-00002
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,liefde;relaties,fictie,Nederlands,219.0,662.0,MBO/NT2,CAT-00003
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,Boek,3F,ontmoeting;verbondenheid,fictie,Nederlands,288.0,748.0,MBO/NT2,CAT-00004
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,Boek,2F,geheimen;schuld,fictie,Nederlands,358.0,196.0,MBO/NT2,CAT-00005


In [ ]:
#in de "type materiaal" mist er 1 waarde waarvan we op hebben gezocht wat het is en dat invullen
df["Type materiaal"] = df["Type materiaal"].fillna("Boek")
df.head(70)

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,Themas_boek,genre,taal,aantal_paginas,geschatte_leestijd_min,doelgroep,cataloguscode
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,boek,3F,WOII;onderduik,fictie,Nederlands,265.0,556.0,MBO,CAT-00001
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,spanning;stalking,fictie,Nederlands,171.0,184.0,MBO/NT2,CAT-00002
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,liefde;relaties,fictie,Nederlands,219.0,662.0,MBO/NT2,CAT-00003
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,Boek,3F,ontmoeting;verbondenheid,fictie,Nederlands,288.0,748.0,MBO/NT2,CAT-00004
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,Boek,2F,geheimen;schuld,fictie,Nederlands,358.0,196.0,MBO/NT2,CAT-00005
...,...,...,...,...,...,...,...,...,...,...,...,...
65,Moordonderzoek,Malcolm Rose,Interactieve detective waarin de lezer aanwijz...,Boek,2F,NaN,fictie,Nederlands,NaN,604.0,MBO/NT2,CAT-00066
66,Zonder spijt,Pieter Aspe,Spannende detectiveroman waarin een onderzoek ...,Boek - roman,3F,NaN,fictie,Nederlands,113.0,129.0,MBO/NT2,CAT-00067
67,Het geluid van de stilte,Inez Weski,"Persoonlijk relaas over rechtspraak, media-aan...",Boek,3F+,NaN,NaN,Nederlands,200.0,617.0,MBO,CAT-00068
68,De zaak Ridouan T.,Hans Werdmolder,Non-fictieboek over georganiseerde criminalite...,Boek,3F,NaN,fictie,NL,143.0,230.0,MBO/NT2,CAT-00069


In [35]:
lege_rijen = df[
    df["Niveau"].isna() |
    df["Niveau"].astype(str).str.strip().eq("")
]

lege_rijen.head(10)

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,Themas_boek,genre,taal,aantal_paginas,geschatte_leestijd_min,doelgroep,cataloguscode,doelgroep_extra
37,Libello voor mannen,NaN,Toegankelijk tijdschrift voor laaggeletterde l...,tijdschrift,NaN,NaN,fictie,Nederlands,354.0,837.0,NaN,CAT-00038,NaN
38,Libello voor vrouwen,NaN,Toegankelijk tijdschrift voor laaggeletterde l...,tijdschrift,NaN,NaN,fictie,Nederlands,153.0,248.0,NaN,CAT-00039,NT2
39,De Startkrant,NaN,Eenvoudig geschreven krant met actueel nieuws ...,krant (papier),NaN,NaN,fictie,Nederlands,311.0,100.0,NaN,CAT-00040,NaN
40,De Startweekkrant,NaN,Wekelijkse digitale krant met begrijpelijke ar...,digitale krant,NaN,NaN,fictie,Nederlands,244.0,386.0,NaN,CAT-00041,NT2
156,"Tekorten zorgpersoneel nemen sneller toe, 'hoo...",NOS,"Zorg, arbeidsmarkt",online artikel,NaN,https://nos.nl/artikel/2548636-tekorten-zorgpe...,informatief,Nederlands,2.0,7.0,NaN,CAT-00157,NaN
157,Nieuwe generatie fysiotherapeuten valt uit na ...,Sander Zurhake (NOS),"Fysiotherapie, zorg",online artikel,NaN,https://nos.nl/artikel/2476626-nieuwe-generati...,informatief,Nederlands,1.0,14.0,NaN,CAT-00158,NT2


In [23]:
#Missing values in de "Niveau" kolom vullen met "Onbekend"
df["Niveau"] = df["Niveau"].fillna("Onbekend")
df.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,Themas_boek,genre,taal,aantal_paginas,geschatte_leestijd_min,doelgroep,cataloguscode
0,Het dagboek van Anne Frank,Anne Frank,Dagboek van een Joods meisje dat tijdens de Tw...,boek,3F,WOII;onderduik,fictie,Nederlands,265.0,556.0,MBO,CAT-00001
1,Blauw Water,S. van der Vlugt,Spannende thriller waarin een vrouw wordt geco...,Boek,3F,spanning;stalking,fictie,Nederlands,171.0,184.0,MBO/NT2,CAT-00002
2,Boer vindt vrouw,H. van der Werf,"Toegankelijk verhaal over liefde, relaties en ...",Boek,2F,liefde;relaties,fictie,Nederlands,219.0,662.0,MBO/NT2,CAT-00003
3,Cadeau voor u!,Harry Mulisch,Bundel toegankelijke verhalen over verrassende...,Boek,3F,ontmoeting;verbondenheid,fictie,Nederlands,288.0,748.0,MBO/NT2,CAT-00004
4,De biecht,Mickey Roorda,Verhaal waarin verborgen geheimen en persoonli...,Boek,2F,geheimen;schuld,fictie,Nederlands,358.0,196.0,MBO/NT2,CAT-00005


In [28]:
lege_rijen = df[
    df["doelgroep"].isna() |
    df["doelgroep"].astype(str).str.strip().eq("")
]

lege_rijen.head()

,Titel,Auteur,Korte omschrijving,Type materiaal,Niveau,Themas_boek,genre,taal,aantal_paginas,geschatte_leestijd_min,doelgroep,cataloguscode
37,Libello voor mannen,Onbekend,Toegankelijk tijdschrift voor laaggeletterde l...,tijdschrift,Onbekend,NaN,fictie,Nederlands,354.0,837.0,,CAT-00038
39,De Startkrant,Onbekend,Eenvoudig geschreven krant met actueel nieuws ...,krant (papier),Onbekend,NaN,fictie,Nederlands,311.0,100.0,,CAT-00040
156,"Tekorten zorgpersoneel nemen sneller toe, 'hoo...",NOS,"Zorg, arbeidsmarkt",online artikel,Onbekend,https://nos.nl/artikel/2548636-tekorten-zorgpe...,informatief,Nederlands,2.0,7.0,,CAT-00157
